In [ ]:
#DEG Muscat (R)

In [ ]:
library(dplyr)
library(ggplot2)
library(limma)
library(muscat)
library(purrr)
library(Seurat)
library(SingleCellExperiment)

In [ ]:
obj<-readRDS("~/John_AML/Seurat_batch_rawcount_0809.rds")
sce$condition_c2<-"test"

sce@meta.data[sce$sample%in%c('P12_C1D1','P03_C1D1','P01_C1D1','P17_C1D1',
                              'P18_C1D1','6P11_C1D1','P09_C1D8'),]$condition_c2<-"pre_R"
sce@meta.data[sce$sample%in%c('P17_C1D1','P18_C1D1','P09_C1D8','P02_C1D1'),]$condition_c2<-"pre_NR"
                                                                                                                                                    
                                                                                                                                                    
                                                                                                                                                    
sce@meta.data[sce$sample%in%c('P12_C6D8','P12_C7D1','P03_C7D22','P01_C7D1',
                              'P17_C7D1','P18_C7D1','6P11_C6D8','P12_C7D22','P03_C12D29','P09_C12D29','P01_C12D29'),]$condition_c2<-"post_R"

sce@meta.data[sce$sample%in%c('P09_C7D1','P02_C7D1','P02_Progression','P17_Progression','P18_C7D22'),]$condition_c2<-"post_NR"

obj<-subset(obj,subset=condition_c2%in%c("pre_NR","post_NR"))

counts <- obj@assays$RNA@counts 
metadata<-obj@meta.data[,c("condition_c2","sample","celltype_0527","patient")]

sce <- SingleCellExperiment(assays = list(counts = counts), 
                           colData = metadata)
sce <- sce[rowSums(counts(sce) > 0) > 0, ]

library(scater)
qc <- perCellQCMetrics(sce)

# remove cells with few or many detected genes
ol <- isOutlier(metric = qc$detected, nmads = 2, log = TRUE)
sce <- sce[, !ol]

(sce <- prepSCE(sce, 
    kid = "annotation", # subpopulation assignments
    gid = "condition_c2",  # group IDs (ctrl/stim)
    sid = "sample", # sample IDs (ctrl/stim.1234)
    drop = TRUE))  # drop all other colData columns

pb <- aggregateData(sce,
    assay = "counts", fun = "sum",
    by = c("cluster_id", "sample_id"))
# one sheet per subpopulation
#assayNames(pb)  # cluster

res <- pbDS(pb, verbose = FALSE,min_cells = 3)

for (i in (1:13)){
write.csv(res$table$`post_NR-pre_NR`[i],paste0("diff_genes_",celltype[i],"_NR_0912.csv"))}
for (i in (1:13)){
write.csv(res$table$`post_R-pre_R`[i],paste0("diff_genes_",celltype[i],"_R_0912.csv"))}

In [ ]:
# GSEApy (python)

In [1]:
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
from gseapy import Msigdb

gene_sets="h.all.v2024.1.Hs.symbols.gmt"

celltype = ['CD4_Naïve', 'Trm', 'CD4 TEM', 'GZMH+ Effector_Memory_2', 'GZMH+ Effector_Memory_1 MAIT', 'TEMRA', 'TEMRA_NK-like',          
    'CD4_Cytotoxic', 'CD4 TCM', 'GZMK+ Effector_Memory', 'Treg', 'gdT']

for a in range(2):
    for i in range(13):
        filename = f"~/John_AML/analysis_new/pseudobulk/diff_genes_{celltype[i]}_{comparison[a]}_0912.csv"
        rnk = pd.read_csv(filename, header=0, index_col=1, sep=",")
        pre_res = gp.prerank(rnk=rnk.sort_values(by=rnk.columns[2]).iloc[:,2], # or rnk = rnk,
                     gene_sets=gene_sets,
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )
        out_dir=f"~/John_AML/analysis_new/pseudobulk/GSEApy/GSEA_hallmark_{celltype[i]}_{comparison[a]}_ranked_0912.csv"
        pre_res.res2d.to_csv(out_dir)

NameError: name 'comparison' is not defined

In [ ]:
#Plotting (R)

In [ ]:
NR_data <- read.csv("pseudobulk/GSEApy/GSEA_hallmark_GZMK+ Effector_Memory_NR_ranked_0912.csv",row.names=1)
R_data <- read.csv("pseudobulk/GSEApy/GSEA_hallmark_GZMK+ Effector_Memory_R_ranked_0912.csv",row.names=1)

NR_data<-NR_data[,c(2,4,5)]
R_data<-R_data[,c(2,4,5)]
rownames(NR_data)<-NR_data$Term
NR_data$group<-"NR"
NR_data<-NR_data[NR_data$Term %in%c('HALLMARK_INTERFERON_GAMMA_RESPONSE'
                              , 'HALLMARK_INTERFERON_ALPHA_RESPONSE'
                              , 'HALLMARK_TNFA_SIGNALING_VIA_NFKB','HALLMARK_INFLAMMATORY_RESPONSE'),]
rownames(R_data)<-R_data$Term
R_data$group<-"R"
R_data<-R_data[R_data$Term %in%c('HALLMARK_INTERFERON_GAMMA_RESPONSE'
                              , 'HALLMARK_INTERFERON_ALPHA_RESPONSE'
                              , 'HALLMARK_TNFA_SIGNALING_VIA_NFKB','HALLMARK_INFLAMMATORY_RESPONSE'),]

df<-rbind(R_data,NR_data)
df$log10p<- (-1)*log10(df$NOM.p.val)

library(ggnewscale)
library(ggplot2)

ggplot(data = subset(df, group == "R"),
aes(x = as.numeric(factor(Term)) - 0.2, y = NES, fill = log10p)) +
geom_col(width = 0.35) +
scale_fill_gradient(low = "#7A9BC1", high = "#0D3A5E", name = "Responder",
breaks=c(1,2,3),limits=c(0,3))+
ggnewscale::new_scale_fill() +
geom_col(data = subset(df,group == "NR"), width = 0.35,
aes(x = as.numeric(factor(Term)) + 0.2, y = NES, fill = log10p),) +
scale_fill_gradient(low = "#FFC680", high = "#FE9003", name = "Non-Responder",
breaks=c(1,2,3),limits=c(0,3))+ #breaks in the scale bar
scale_x_continuous(breaks = seq_along(levels(factor(df$Term))),
labels = levels(factor(df$Term)), name = "Term") +
scale_y_continuous(breaks=c(-2,-1,0,1,2))+ylim(-2.5,2.5)+
theme_minimal(base_size = 16) +
theme(panel.grid.major.x = element_blank())+coord_flip()+geom_hline(yintercept = 0, linetype = 1)+theme_classic()+
facet_wrap(~cluster,scales = "free")+
theme(strip.background = element_blank(), # Remove the box around the title
strip.text = element_text(face = "bold"), # Optional: make the title bold
axis.text = element_text(size = 10), # Adjust axis text size
axis.title = element_text(size = 16))
ggsave("test.pdf",width = 20,height=4)
